In [2]:
import os
import yaml
import numpy as np
from scipy.stats import spearmanr
import datasets
import pathlib

def get_cluster_indices(num_clusters):
    cluster_indices = {}
    
    for cluster_id in range(num_clusters):
        cluster_file = os.path.join(base_dir, f"cluster_{cluster_id}.npy")
        if os.path.exists(cluster_file):
            cluster_i = np.load(cluster_file)
            if cluster_i.size > 0:
                try:
                    indices = cluster_i[:, 1].astype("int32")
                except:
                    indices = cluster_i.astype("int32")
                cluster_indices[cluster_id] = indices
        else:
            print(f"Cluster file {cluster_file} does not exist.")
    
    return cluster_indices

# Load average reward and cluster chose ratio
# base_dir = "/data/datasets/hf_cache/data/fineweb/sample-350BT/bge_micro_cluster_info/0/sorted_clusters/"
base_dir = "/data/datasets/hf_cache/data/fineweb/sample-350BT/train_bge_micro_embeddings_index_1000/cluster_indices/"
#get metrics
metric_name = "10000-data_influence_model-flan-prediction" #"less_full", "oracle"
average_reward = np.load(pathlib.Path(base_dir, f"average_reward_{metric_name}.npy"))
print(average_reward.shape)
cluster_chose_ratio = np.load(f"../out/{metric_name}/cluster_chose_ratio.npy")
print(cluster_chose_ratio.shape)

metric_split_path = os.path.join("/data/datasets/hf_cache/out/pythia-1b/fineweb/sample-350BT/train/0", metric_name)
max_splits = len([name for name in os.listdir(metric_split_path) if os.path.isdir(os.path.join(metric_split_path, name))])
metrics_dataset = datasets.concatenate_datasets(
    [
        datasets.load_from_disk(
            os.path.join(metric_split_path, str(i))
        )
        for i in range(max_splits)
    ]
)
print(metrics_dataset)
metrics = np.array(metrics_dataset["prediction"]).reshape(-1)
metrics = metrics[:1638400]
print(">> Metrics shape:", metrics.shape)
    

new_metrics = metrics.copy()
num_clusters = 1000
# Get cluster indices
cluster_indices = get_cluster_indices(num_clusters)
print({k: len(v) for k, v in cluster_indices.items()})

# Calculate new metrics based on cluster chose ratio
for cluster_id in range(num_clusters):
    if cluster_id in cluster_indices:
        indices = cluster_indices[cluster_id]
        valid_indices = indices[indices < len(metrics)]  # Only consider indices within len(metrics)
        if len(valid_indices) > 0:
            new_metrics[valid_indices] = np.mean(metrics[valid_indices])

# Calculate Spearman correlation
correlation, _ = spearmanr(average_reward, cluster_chose_ratio)
print(f"Spearman correlation: {correlation}")

# Print Spearman correlations for specified ranges
print(spearmanr(metrics[:1000], new_metrics[:1000]))
print(spearmanr(metrics, new_metrics))

(1000,)
(1000,)
Dataset({
    features: ['index', 'prediction'],
    num_rows: 8724336
})


>> Metrics shape: (1638400,)
{0: 30630, 1: 4049, 2: 23281, 3: 17713, 4: 15469, 5: 24348, 6: 18741, 7: 11601, 8: 26183, 9: 6065, 10: 27726, 11: 26132, 12: 29984, 13: 19497, 14: 22436, 15: 28014, 16: 21032, 17: 26948, 18: 17071, 19: 11955, 20: 13305, 21: 15081, 22: 13710, 23: 3019, 24: 23344, 25: 12607, 26: 14027, 27: 22760, 28: 25007, 29: 19597, 30: 31721, 31: 15382, 32: 19113, 33: 22227, 34: 6112, 35: 8583, 36: 17897, 37: 9992, 38: 25124, 39: 20493, 40: 29870, 41: 20594, 42: 10819, 43: 19017, 44: 11337, 45: 7472, 46: 23898, 47: 10205, 48: 19727, 49: 27956, 50: 24672, 51: 13899, 52: 12280, 53: 20936, 54: 11116, 55: 29325, 56: 14321, 57: 17790, 58: 28139, 59: 11998, 60: 28496, 61: 100, 62: 22208, 63: 8061, 64: 23468, 65: 22034, 66: 16592, 67: 5662, 68: 17756, 69: 16096, 70: 7541, 71: 8782, 72: 17009, 73: 5434, 74: 12892, 75: 7715, 76: 16879, 77: 1102, 78: 23262, 79: 15472, 80: 21037, 81: 16189, 82: 26684, 83: 22766, 84: 13832, 85: 18983, 86: 24734, 87: 8140, 88: 16181, 89: 21229, 90: 113